# 03 · Retrieval-augmented generation

**Session 01 · Building AI applications for research and data · notebook 3 of 6**

## Objective

Notebook 02 built the vector store and left two things open: a search for the latest figures puts the 2016 summary first, and nothing yet uses the date. This notebook puts the store to work. The application searches it for the chunks closest to a question, gives them to the model as context, and the model answers from them. A chunk that the search returns and the model reads is called a passage.

This pattern has a name: **retrieval-augmented generation**, or **RAG**. *Retrieval* is the search of the store, *augmented* means the search results are added to what the model is given, and *generation* is the model writing the answer. RAG has two halves: notebook 02 prepared the collection, and this notebook uses it to answer. The rest of the series builds on it. This notebook builds the second half, then shows three ways it goes wrong, and what fixes each.

## The application after this notebook

Diagram: the question goes to retrieval, a search of the vector store that returns the closest chunks as passages; the passages and the question go into the prompt, then to generation, where the model writes an answer with citations.

Retrieval (the search) and the passages it returns are what this notebook adds, and the model's step becomes generation: together, the RAG pattern.

## What you will see

Four short scenes:

1. The question from notebook 01 answered from the store. The answer cites its passages, but half of the question goes unanswered.
2. Why the economy briefs never reached the model, and a fix that breaks the question into its parts and searches once for each.
3. The 2016 summary outranking the 2026 briefs, and the date from notebook 02 removing it.
4. A question the collection cannot answer still gets passages, and how to test a search without calling the model.

## How to run it

Run the cells in order. The model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. The searches, rankings and tables are computed when you run the cell.

The next cell installs the session's tools and loads its helper module. It takes about a minute the first time, and you do not need to read it.

In [ ]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/rafmacalaba/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-session-01-v2"
SESSION = "training/session-01-building-ai-applications-v2/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

The next cell builds the same store as notebook 02, in three lines. Each Colab notebook starts with an empty session, so the store from notebook 02 is not there to reuse; the three lines rebuild it. The cell also sets up the application: the instructions, a prompt, and two functions, one that searches the store and asks the model, and one that only searches. `k` is the number of passages a search returns; the model reads only those.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

docs = w.load_documents()
chunks = w.split_sections(docs)
store = w.build_store(chunks)
print(len(chunks), "chunks in the store, the same collection as in notebook 02")

INSTRUCTIONS = """You are a research assistant for development data.
Answer only from the supplied passages. After each figure, cite the passage in the square-bracket form given.
If the passages do not contain what is needed, say so."""

prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])


def make_rag(k=4, where=None):
    """RAG (retrieval-augmented generation): search the store, put the passages in the prompt, ask the model."""
    retrieve = RunnableLambda(lambda q: store.similarity_search(q, k=k, filter=where))
    return {"context": retrieve | w.format_context, "question": RunnablePassthrough()} | prompt | model


def hits(question, k=4, where=None):
    return store.similarity_search_with_score(question, k=k, filter=where)

## 1. The question from notebook 01, answered from the store

The question is the main one from notebook 01. The search returns the four chunks closest to it, and they become the passages the model reads. The cell prints the question, then the passages, so that the evidence the model saw is visible, then the model's answer.

In [ ]:
w.show("Question: " + w.MAIN_QUESTION)
print()
w.print_hits(hits(w.MAIN_QUESTION, k=4))
print()
w.use("nb03_rag_main")
w.show(make_rag(k=4).invoke(w.MAIN_QUESTION))

The answer cites its passages, and it is honest: it says the passages hold no figures on GDP or growth, so half of the question cannot be answered. The briefs with those figures are in the store. Why did they not come back?

## 2. Why half the question was lost

The question asks about internet access and economic development. The search compares the whole question with every chunk, and keeps the four closest. The cell ranks the chunks for the question, coloured by topic, with a line after the first four, the ones that go to the model.

In [ ]:
topic = {"kenya-digital": "internet", "rwanda-digital": "internet", "regional-2016": "internet", "internet-indicator": "internet",
         "kenya-economy": "economy", "rwanda-economy": "economy", "coffee": "other", "mobile-money": "other"}
legend = {"internet": ("internet use", "#2f5f9e"), "economy": ("economy", "#d9822b"), "other": ("other topics", "#8a94a3")}

ranked = hits(w.MAIN_QUESTION, k=10)
w.show_ranking([(w.citation(d), score, topic[d.metadata["id"]]) for d, score in ranked], k=4, legend=legend, question=w.MAIN_QUESTION)

The top four are three internet chunks, one of them the 2016 summary, and the note on mobile money. The first economy chunk comes fifth, just below the line (0.570 against 0.591), and the one with Kenya's GDP per capita comes ninth. The search does not know that the question has two halves: it ranks chunks by their closeness to the whole question. Raising the number of passages is a poor fix: Kenya's GDP per capita appears only at nine passages, together with the coffee note and others, and every passage costs, as notebook 01 showed.

The fix is on the search side: **break the question down into its parts, search once for each part, and merge the passages.** This question has two parts, internet use and the economy, so it becomes two smaller questions. Libraries can have a model write these smaller questions; here the two are written by hand so that the mechanism is visible. The cell shows the original search and the merged one side by side, and marks which of the four figures the question needs each of them brings back. The fix is not free: two searches of four chunks each give eight different passages, against four before, and the 2016 summary and the mobile-money note come along with them. The model ignores both here, but section 3 shows what happens when it does not.

In [ ]:
# the question broken into its two parts: one smaller question for each
sub_queries = ["internet use in Kenya and Rwanda, recent figures", "GDP per capita and economic growth in Kenya and Rwanda, recent figures"]
merged = []
for q in sub_queries:
    for d in store.similarity_search(q, k=4):
        if d not in merged:
            merged.append(d)

one_search = [d for d, _ in hits(w.MAIN_QUESTION, k=4)]
needed = next(item for item in w.load_questions() if item["id"] == "q1")["expected_numbers"]
countries = {"KEN": "Kenya", "RWA": "Rwanda"}
names = {"IT.NET.USER.ZS": "internet use", "NY.GDP.PCAP.CD": "GDP per capita"}
checks = [f"{countries[n['country']]} {names[n['indicator']]}, {n['year']}: {n['value']:,}" for n in needed]


def column(title, tone, found):
    text = " ".join(d.page_content for d in found)
    return {"title": title, "tone": tone, "hits": [(w.citation(d), d.page_content) for d in found],
            "found": [w.numbers_present(text, [n])[0] == 1 for n in needed]}


w.show_search_comparison(w.MAIN_QUESTION, [column("One search for the whole question: 4 passages", "bad", one_search),
                                           column(f"The question broken into two, one search each, merged: {len(merged)} passages", "good", merged)], checks)
print()
w.use("nb03_rag_two_queries")
w.show((prompt | model).invoke({"context": w.format_context(merged), "question": w.MAIN_QUESTION}))

The merged passages bring back all four figures, where the single search brought two, and the answer covers both halves of the question.

## 3. The 2016 summary outranks the briefs

Notebook 02 ended with the 2016 summary ranking first. Here is what that costs. The question is "What are the latest available figures for internet use in Kenya and Rwanda?", and the 2016 summary contains the words "latest available figures". With two passages supplied, the Kenya figure comes from the 2016 summary (a 2015 value) and the Rwanda figure from a 2026 brief (a 2024 value), and the model reports both faithfully.

In [ ]:
q = "What are the latest available figures for internet use in Kenya and Rwanda?"
w.print_hits(hits(q, k=2))
print()
w.use("nb03_rag_outdated")
w.show(make_rag(k=2).invoke(q))

The answer is faithful to its evidence and wrong for the reader: a 2015 figure presented as the latest. Nothing in the generation step can fix this; the evidence was selected badly.

## 4. Three ideas for fixing outdated evidence

The fix is on the search side. Three ideas:

1. **Filter on the date.** Tell the search to ignore documents dated before 2020. Notebook 02 kept each document's date as metadata, for this moment.
2. **Reword the question** to look like the passage it should find, for example "internet use 2024, percent of population". It often helps, but not here.
3. **Sort the results by date** after the search, when the store has no filter. It does the same job as the filter; it is not shown here.

The notebook tries the first two. The first cell compares the same search without and with the date filter, and marks which of the two figures the question needs turn up.

In [ ]:
since_2020 = lambda d: str(d.metadata.get("date", "")) >= "2020"
expected = next(item for item in w.load_questions() if item["id"] == "q8")["expected_numbers"]
countries = {"KEN": "Kenya", "RWA": "Rwanda"}
checks = [f"{countries[e['country']]}, {e['year']}: {e['value']}" for e in expected]


def search(title, tone, where=None):
    found = store.similarity_search(q, k=2, filter=where)
    text = " ".join(h.page_content for h in found)
    return {"title": title, "tone": tone, "hits": [(w.citation(h), h.page_content) for h in found],
            "found": [w.numbers_present(text, [e])[0] == 1 for e in expected]}


w.show_search_comparison(q, [search("No filter", "bad"), search("Date filter: 2020 or later", "good", since_2020)], checks)

With the filter, the 2016 summary is gone and both figures come back. The model's answer from the filtered passages:

In [ ]:
w.use("nb03_rag_filtered")
w.show(make_rag(k=2, where=since_2020).invoke(q))

In [ ]:
w.print_hits(hits("Kenya and Rwanda internet use 2024, percent of population", k=2))

## 5. A question outside the collection

A search always returns the closest chunks, even when none of them is close. Compare a question the collection can answer with one it cannot. The score shows how close a passage is to the question: higher is closer.

In [ ]:
for q2 in ["What is the population of Rwanda?", "What was inflation in Uganda in 2023?"]:
    print(q2)
    w.print_hits(hits(q2, k=3))
    print()

Both questions get three passages. The scores are lower for the question outside the collection, and a cut-off score could separate the two here, but a cut-off has to be set from examples on the embedding model in use. The second safeguard is the instruction from notebook 01 that tells the model to say when the passages do not answer the question; notebook 05 shows it at work. Applications use both.

## 6. Testing the search without the model

A search can be tested before any model is involved. This is the retrieval half of RAG on its own, and the model is the generation half. For each of the eight evaluation questions, the cell makes two checks: did the passages include an expected document (one of them is enough), and do they contain the numbers an answer needs? `K` is the number of passages the search returns.

In [ ]:
K = 4   # passages returned by the search: try 2, then 8

rows = []
for item in w.load_questions():
    passages = store.similarity_search(item["question"], k=K)
    text = " ".join(p.page_content for p in passages)
    needed = item.get("expected_numbers", [])
    found = sum(w.numbers_present(text, [n])[0] for n in needed)
    rows.append((item["id"], item["category"],
                 "hit" if w.retrieval_hit(passages, item["expected_documents"]) else "miss",
                 f"{found}/{len(needed)}" if needed else "none needed"))
w.show_table(["id", "category", "expected document", "needed numbers"], rows, source=f"The search returns {K} passages per question.")

Question q1, the main question, passes the first check, though only the two internet documents of its four expected ones came back, and fails the second at `K = 4`: documents of the right kind came back, but only two of the four numbers the answer needs. That is a retrieval problem, and no instruction to the model can fix it. When both checks pass and the answer is still wrong, the fault is in the generation: the instructions or the model, which notebook 06 measures. Keeping the two apart is what makes each one possible to improve.

## Key points

- Retrieval-augmented generation (RAG), introduced at the top: search the store for the chunks closest to the question, give them to the model as passages, and have it answer from them, citing each.
- What the search returns decides what the answer can say. For the main question the economy chunks ranked just below the line and never reached the model; breaking the question into its parts and searching once for each fixed it.
- A search matches meaning, not age. The 2016 summary outranked the 2026 briefs, and rewording the question did not help; the date kept as metadata, used as a filter, did.
- A search always returns something, even for a question the collection cannot answer. Two safeguards: a cut-off score set from examples, and an instruction telling the model to say when the passages do not answer.
- A search can be tested without the model: check that the right documents and the needed numbers are among the passages. A miss there cannot be fixed by changing the instructions.

## Exercise

1. Look at the ranking in section 2. Which chunk takes fourth place, and is it a good use of one of the four slots?
2. In the cell of section 6, set `K` to `2`, run it, then set it to `8` and run it again. Which questions lose numbers at 2, and which gain at 8? At 8, does every question find all the numbers it needs?
3. Set `K` to `2` and look at q8, the question from section 3. Why does it pass at 4 and not at 2?

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.